# Bank Marketing Dataset

The dataset is related with direct marketing campaigns (phone calls) of a Portuguese
banking institution. The goal is to predict if the client will subscribe a term deposit. The
attributes information of the dataset as follows:

## Import Libraries

In [53]:
import pandas as pd
import numpy as np
import joblib as jb

## Import Data

In [54]:
import csv
global dataset
try:
    with open('../data/bank-additional-full.csv', 'r', newline='') as file:
        sample = file.read(1024)  # Read a sample from the file
        detected_delimiter = csv.Sniffer().sniff(sample).delimiter
except Exception as e:
    print("Error: ", e)

# Load the dataset
try:
    dataset = pd.read_csv('../data/bank-additional-full.csv', delimiter=detected_delimiter)
except Exception as e:
    print("Error: ", e)

#### Drop single row contains more than 2 "unknown" values

In [55]:
drop_columns = ["contact", "day_of_week", "month", "duration", "cons.conf.idx", "nr.employed", "emp.var.rate", "y",
                "cons.price.idx", "euribor3m", "nr.employed", ]

dataset = dataset[dataset.apply(lambda row: (row == "unknown").sum(), axis=1) <= 3]
X = dataset.drop(columns=drop_columns).values
Y = dataset.iloc[:, -1].values

In [56]:
print(X[1])

[57 'services' 'married' 'high.school' 'unknown' 'no' 'no' 1 999 0
 'nonexistent']


In [57]:
print(Y[:10])

['no' 'no' 'no' 'no' 'no' 'no' 'no' 'no' 'no' 'no']


## Encode Categorical Data

In [58]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

ct = ColumnTransformer(transformers=[("encoder", OneHotEncoder(sparse_output=False), [1, 2, 3, 4, 5, 6, 10])],
                       remainder="passthrough", )
X = np.array(ct.fit_transform(X))

In [59]:
print(X[1])

[0.0 0.0 0.0 0.0 0.0 0.0 0.0 1.0 0.0 0.0 0.0 0.0 0.0 1.0 0.0 0.0 0.0 0.0
 0.0 1.0 0.0 0.0 0.0 0.0 0.0 1.0 0.0 1.0 0.0 0.0 1.0 0.0 0.0 0.0 1.0 0.0
 57 1 999 0]


## Label Defendant Variable

In [60]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
Y = le.fit_transform(y=Y)

In [61]:
print(Y[:10])

[0 0 0 0 0 0 0 0 0 0]


## Splitting Train Test

In [62]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=.2, train_size=.8, random_state=50)

In [63]:
print(x_train[1])

[1.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 1.0 0.0 0.0 0.0
 0.0 0.0 0.0 1.0 0.0 0.0 1.0 0.0 0.0 0.0 0.0 1.0 0.0 0.0 1.0 0.0 1.0 0.0
 27 1 999 0]


In [64]:
print(y_train[:10])

[0 0 0 0 1 0 0 0 0 0]


In [65]:
print(x_test[1])

[0.0 0.0 1.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 1.0 0.0 0.0 0.0 0.0
 0.0 0.0 0.0 1.0 0.0 0.0 1.0 0.0 0.0 0.0 0.0 1.0 1.0 0.0 0.0 0.0 1.0 0.0
 41 1 999 0]


In [66]:
print(y_test[:10])

[0 0 0 0 0 0 0 0 0 0]


## Feature Scaling

In [67]:
from sklearn.preprocessing import StandardScaler

sc = StandardScaler()
x_train[:, -4:-1] = sc.fit_transform(x_train[:, -4:-1])
x_test[:, -4:-1] = sc.transform(x_test[:, -4:-1])

In [68]:
print(x_train[1])

[1.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 1.0 0.0 0.0 0.0
 0.0 0.0 0.0 1.0 0.0 0.0 1.0 0.0 0.0 0.0 0.0 1.0 0.0 0.0 1.0 0.0 1.0 0.0
 -1.2528674653747303 -0.5657009896435832 0.19572160745675077 0]


In [69]:
print(x_test[1])

[0.0 0.0 1.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 1.0 0.0 0.0 0.0 0.0
 0.0 0.0 0.0 1.0 0.0 0.0 1.0 0.0 0.0 0.0 0.0 1.0 1.0 0.0 0.0 0.0 1.0 0.0
 0.09416877427082597 -0.5657009896435832 0.19572160745675077 0]


## Train Logistic Regression

In [70]:
from sklearn.metrics import classification_report

In [71]:
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(max_iter=10000)
logistic_model.fit(x_train, y_train)

LogisticRegression(max_iter=10000)

In [72]:
y_predict = logistic_model.predict(x_test)

In [73]:
print("Classification Report\n")
print(classification_report(y_test, y_predict))

Classification Report

              precision    recall  f1-score   support

           0       0.91      0.99      0.94      7316
           1       0.63      0.19      0.29       918

    accuracy                           0.90      8234
   macro avg       0.77      0.59      0.62      8234
weighted avg       0.88      0.90      0.87      8234



## Save the Model

In [74]:
jb.dump(logistic_model,"../build_models/logistic_model.pkl")

['../build_models/logistic_model.pkl']

## Train Support Vector Machine

In [75]:
from sklearn.svm import SVC

svc_model = SVC()
svc_model.fit(x_train, y_train)

SVC()

In [76]:
y_predict_svc = svc_model.predict(x_test)

In [77]:
print("Classification Report\n")
print(classification_report(y_test, y_predict_svc))

Classification Report

              precision    recall  f1-score   support

           0       0.91      0.99      0.94      7316
           1       0.63      0.18      0.28       918

    accuracy                           0.90      8234
   macro avg       0.77      0.58      0.61      8234
weighted avg       0.88      0.90      0.87      8234



## Save the Model

In [78]:
jb.dump(svc_model, "../build_models/svc_model.pkl")

['../build_models/svc_model.pkl']